# Schritt 1: Vegane Restaurants in der Schweiz (OpenStreetMap / Overpass API)

Dieses Notebook
- holt Restaurants, Cafés und Fast-Food-Lokale mit veganem Angebot aus OpenStreetMap (ganze Schweiz),
- unterscheidet **rein vegan** (`vegan_only`) und **mit veganem Angebot** (`vegan_option`),
- speichert die Rohdaten als JSON und die bereinigte Tabelle als CSV und in einer SQLite-Datenbank.

Hinweis: Wenn ein `JSONDecodeError` oder Timeout auftritt, ist der Overpass-Server überlastet. Einfach nochmals ausführen.

In [ ]:
import json, time, sqlite3
import requests
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## Overpass-Abfrage (ganze Schweiz)

In [ ]:
overpass_urls = [
    'https://overpass-api.de/api/interpreter',
    'https://overpass.osm.ch/api/interpreter',
]

query = """
[out:json][timeout:180];
area["ISO3166-1"="CH"][admin_level=2]->.ch;
(
  nwr["amenity"~"^(restaurant|cafe|fast_food)$"]["diet:vegan"~"^(yes|only)$"](area.ch);
  nwr["amenity"~"^(restaurant|cafe|fast_food)$"]["cuisine"~"vegan"](area.ch);
);
out center tags;
"""

data = None
for attempt in range(5):
    for url in overpass_urls:
        try:
            r = requests.post(url, data={'data': query}, timeout=200)
            r.raise_for_status()
            data = r.json()
            print(f'OK via {url} (Versuch {attempt + 1})')
            break
        except Exception as e:
            print(f'Fehler bei {url}: {e}')
            time.sleep(10)
    if data:
        break

print('Anzahl Elemente:', len(data['elements']) if data else 0)

## Rohdaten speichern

In [ ]:
with open('vegan_restaurants_ch_raw.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False)

## In DataFrame umwandeln

Definition:
- `vegan_only`: `diet:vegan=only` oder `cuisine` enthält *vegan*
- `vegan_option`: `diet:vegan=yes` (bietet vegane Gerichte an)

Bei Flächen (Gebäuden) und Relationen liefert Overpass den Mittelpunkt (`center`).

In [ ]:
records = []
for el in data['elements']:
    tags = el.get('tags', {})
    lat = el.get('lat') or el.get('center', {}).get('lat')
    lon = el.get('lon') or el.get('center', {}).get('lon')
    diet = tags.get('diet:vegan', '')
    cuisine = tags.get('cuisine', '')
    records.append({
        'osm_type': el['type'],
        'osm_id': el['id'],
        'lat': lat,
        'lon': lon,
        'name': tags.get('name', 'unknown'),
        'amenity': tags.get('amenity'),
        'cuisine': cuisine,
        'diet_vegan': diet,
        'postcode': tags.get('addr:postcode'),
        'city': tags.get('addr:city'),
    })

df = pd.DataFrame(records)
df['vegan_only'] = (df['diet_vegan'] == 'only') | df['cuisine'].str.contains('vegan', case=False, na=False)
df['vegan_category'] = df['vegan_only'].map({True: 'vegan_only', False: 'vegan_option'})

n_raw = len(df)
df = df.dropna(subset=['lat', 'lon']).drop_duplicates(subset=['osm_type', 'osm_id'])
print(f'Datensätze roh: {n_raw}, nach Bereinigung: {len(df)}')
df.head()

## Erster Überblick

In [ ]:
print(df['vegan_category'].value_counts())
print()
print(df['amenity'].value_counts())
print()
print('Top 10 Orte (Angabe im OSM-Eintrag, oft unvollständig):')
print(df['city'].value_counts().head(10))

## Speichern: CSV und SQLite

In [ ]:
df.to_csv('vegan_restaurants_ch.csv', index=False, encoding='utf-8')

with sqlite3.connect('vegan_project.db') as conn:
    df.to_sql('restaurants', conn, if_exists='replace', index=False)
    print(pd.read_sql('SELECT vegan_category, COUNT(*) AS n FROM restaurants GROUP BY vegan_category', conn))